# LexChain parser benchmark — Docling vs MinerU vs Marker on OHR-Bench (Law)

**Kaggle setup:** GPU T4 x2 accelerator, internet ON.

Flow: **Cell 1** setup → **Cell 2** 3-document smoke test + full-run time estimate → *approve* → **Cell 3** full run (95 docs) → **Cell 4** metrics table + CSV.

**If the session dies:** progress is checkpointed after every document in `/kaggle/working/results`. In a new session, add the previous run's output as an *Input dataset*, then pass `--restore-from /kaggle/input/<slug>/results` in Cell 3.

In [2]:
# Cell 1 — clone repo + set up envs, data, models (~15-25 min first time)
REPO_URL = "https://github.com/MarvinPescos/lexchain-parser-bench.git"

import os
if not os.path.exists("/kaggle/working/lexchain-parser-bench"):
    !git clone {REPO_URL} /kaggle/working/lexchain-parser-bench
%cd /kaggle/working/lexchain-parser-bench
!git pull
!nvidia-smi -L
!bash setup.sh

Cloning into '/kaggle/working/lexchain-parser-bench'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 30 (delta 9), reused 29 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 34.34 KiB | 5.72 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/kaggle/working/lexchain-parser-bench
Already up to date.
GPU 0: Tesla T4 (UUID: GPU-82832b4d-0840-714b-cbb8-0f1a7cf2c23c)
GPU 1: Tesla T4 (UUID: GPU-e8364d0b-79b6-5421-b191-79acda9b8252)
== envs: /kaggle/tmp/envs | data: /kaggle/tmp/ohr_data | hf cache: /kaggle/tmp/hf
== env docling: installing docling
Using CPython 3.11.15
Creating virtual environment at: /kaggle/tmp/envs/docling
Activate with: source /kaggle/tmp/envs/docling/bin/activate
== env mineru: installing mineru[core]
Using CPython 3.11.15
Creating virtual environment at: /kaggle/tmp/envs/mineru
Activate with: source /kaggle/tmp/envs/mineru/bin/activate
== env marker: in

In [ ]:
# Cell 2 — smoke test: 3 smallest PDFs through all 3 tools, then estimate the full run
!python run_benchmark.py --limit 3
!python evaluate.py
!python estimate_runtime.py

**STOP — review the estimate above before running the full benchmark.**

Resuming a previous session? Attach its output as an input dataset and use:
`!python run_benchmark.py --restore-from /kaggle/input/<slug>/results`

In [ ]:
# Cell 3 — full run (resumable; smoke-test docs are skipped automatically)
!python run_benchmark.py

In [ ]:
# Cell 4 — evaluate + paper table (CSV + markdown saved to /kaggle/working/results)
!python evaluate.py

import pandas as pd
from IPython.display import display
display(pd.read_csv("/kaggle/working/results/results_summary.csv"))
print(open("/kaggle/working/results/results.md").read())

---
## Part 2 — Simulated-scanned condition

The Law set is 88 digital-born / 7 natively-scanned PDFs, but LexChain's real workload is mostly scanned. The cells below rasterize **all 95 PDFs at 200 DPI into image-only PDFs** (no text layer, identical filenames → ground truth maps 1:1) and rerun the benchmark with every tool forced through its **OCR path**. Tool configs are identical to Part 1 — the input condition is the only variable.

Results go to `/kaggle/working/results_scanned` (Part 1 results are untouched). Same checkpoint/resume behavior; to resume a dead session: `--restore-from /kaggle/input/<slug>/results_scanned`.

In [3]:
import subprocess
src = subprocess.run(["find","/kaggle/input","-type","d","-name","results"],capture_output=True,text=True).stdout.strip().split("\n")[0]
!cp -r {src} /kaggle/working/
!ls /kaggle/working/results/docling/meta | wc -l   # expect 95

95


In [4]:
# Cell 5 — build the simulated-scanned set (resumable; ~10-20 min, CPU only)
!python make_scanned.py

  ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT.pdf: 3 pages rasterized @ 200 DPI
  ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .pdf: 6 pages rasterized @ 200 DPI
  ADUROBIOTECH,INC_06_02_2020-EX-10.7-CONSULTING AGREEMENT.pdf: 4 pages rasterized @ 200 DPI
  AIRTECHINTERNATIONALGROUPINC_05_08_2000-EX-10.4-FRANCHISE AGREEMENT.pdf: 16 pages rasterized @ 200 DPI
  AMERICASSHOPPINGMALLINC_12_10_1999-EX-10.2-SITE DEVELOPMENT AND HOSTING AGREEMENT.pdf: 5 pages rasterized @ 200 DPI
  ASIANDRAGONGROUPINC_08_11_2005-EX-10.5-Reseller Agreement.pdf: 18 pages rasterized @ 200 DPI
  ArcGroupInc_20171211_8-K_EX-10.1_10976103_EX-10.1_Sponsorship Agreement.pdf: 4 pages rasterized @ 200 DPI
  ArmstrongFlooringInc_20190107_8-K_EX-10.2_11471795_EX-10.2_Intellectual Property Agreement.pdf: 40 pages rasterized @ 200 DPI
  Array BioPharma Inc. - LICENSE, DEVELOPMENT AND COMMERCIALIZATION AGREEMENT.pdf: 107 pages rasterized @ 200 DPI
  BANUESTRAFINANC

In [20]:
# Cell 6 — scanned-condition smoke test (3 docs) + full-run estimate
# OCR mode is typically 3-6x slower than the digital condition — check the
# projection against Kaggle's ~12 h session cap before running Cell 7.
!python run_benchmark.py --condition scanned --limit 3
!python estimate_runtime.py --results-dir /kaggle/working/results_scanned --label scanned

[bench 07:12:30] condition: scanned (pdfs: /kaggle/tmp/ohr_data/pdfs/law_scanned, results: /kaggle/working/results_scanned)
[bench 07:12:30] 3 docs x 3 tools
[bench 07:12:30] GPUs detected: ['0', '1'] -> 2 parallel worker(s)
[bench 07:12:30] docling: worker started on GPU 0 (3 docs pending, restart 0)
[bench 07:12:30] mineru: worker started on GPU 1 (3 docs pending, restart 0)
[bench 07:13:05] docling: all docs processed
[bench 07:13:05] mineru: all docs processed
[bench 07:13:05] marker: worker started on GPU 0 (3 docs pending, restart 0)
[bench 07:14:50] marker: all docs processed
[bench 07:14:50] done in 2.3 min
[bench 07:14:50] docling: {'success': 3}
[bench 07:14:50] mineru: {'success': 3}
[bench 07:14:50] marker: {'success': 3}
=== Full-run estimate [scanned]: 1187 pages, 95 docs ===
docling: 2.4 s/page over 3 doc(s) (model load 19s) -> ~0.8 h alone
mineru: 4.5 s/page over 3 doc(s) (model load 6s) -> ~1.5 h alone
marker: 17.2 s/page over 3 doc(s) (model load 13s) -> ~5.7 h alone


**STOP — review the scanned-condition estimate above before the full OCR run.** If it exceeds the session cap, run Cell 7 in chunks across sessions (checkpointed, resumable via `--restore-from`).

In [5]:
# Cell 7 — full scanned-condition run (resumable; smoke docs skipped)
!python run_benchmark.py --condition scanned

[bench 14:41:07] condition: scanned (pdfs: /kaggle/tmp/ohr_data/pdfs/law_scanned, results: /kaggle/working/results_scanned)
[bench 14:41:07] 95 docs x 3 tools
[bench 14:41:07] GPUs detected: ['0', '1'] -> 2 parallel worker(s)
[bench 14:41:07] docling: worker started on GPU 0 (95 docs pending, restart 0)
[bench 14:41:07] mineru: worker started on GPU 1 (95 docs pending, restart 0)
[bench 15:02:47] mineru: all docs processed
[bench 15:02:47] marker: worker started on GPU 1 (95 docs pending, restart 0)
[bench 15:32:37] docling: all docs processed
[bench 20:42:24] marker: all docs processed
[bench 20:42:24] done in 361.3 min
[bench 20:42:24] docling: {'success': 95}
[bench 20:42:24] mineru: {'success': 95}
[bench 20:42:24] marker: {'success': 95}


In [6]:
!cd /kaggle/working && zip -qr all_results.zip results results_scanned && ls -lh all_results.zip

-rw-r--r-- 1 root root 13M Jul 18 20:47 all_results.zip


In [8]:
# Cell 8 — combined evaluation: digital-born (88) vs natively-scanned (7) vs
# simulated-scanned (95), per tool, + ranking-stability check
!python evaluate.py --detect-scanned
!python compare_conditions.py

import pandas as pd
from IPython.display import display
display(pd.read_csv("/kaggle/working/results/comparison.csv"))
print(open("/kaggle/working/results/comparison.md").read())

detect-scanned: 88 digital-born, 7 natively scanned -> /kaggle/working/results/digital_docs.txt, scanned_docs.txt

--- digital-born (n=88, /kaggle/working/results) ---
[docling] 87 docs, success 100%, NED 0.1512
[mineru] 87 docs, success 100%, NED 0.1400
[marker] 87 docs, success 100%, NED 0.1159

--- natively-scanned (n=7, /kaggle/working/results) ---
[docling] 7 docs, success 100%, NED 0.2583
[mineru] 7 docs, success 100%, NED 0.1969
[marker] 7 docs, success 100%, NED 0.1560

--- simulated-scanned (n=95, /kaggle/working/results_scanned) ---
[docling] 95 docs, success 100%, NED 0.2609
[mineru] 95 docs, success 100%, NED 0.1473
[marker] 95 docs, success 100%, NED 0.1124

| Condition         |   n | Tool    |   NED ↓ |   CER ↓ |   TEDS ↑ |   TEDS-S ↑ |   Read. order ↓ |   s/page ↓ |   Success ↑ |
|:------------------|----:|:--------|--------:|--------:|---------:|-----------:|----------------:|-----------:|------------:|
| digital-born      |  87 | docling |  0.1512 |  0.2130 |   0.3819

,condition,tool,docs,success_rate,ned,cer,teds,teds_s,gt_tables_scored,reading_order,s_per_page,failures
0,digital-born,docling,87,1.0,0.151237,0.213002,0.381896,0.446153,117,0.035873,0.589465,0
1,digital-born,mineru,87,1.0,0.139994,0.204419,0.313841,0.369491,117,0.036763,0.729172,0
2,digital-born,marker,87,1.0,0.115930,0.190663,0.453070,0.528069,117,0.025150,2.358013,0
3,natively-scanned,docling,7,1.0,0.258306,0.270184,0.467282,0.539801,10,0.136178,2.381389,0
4,natively-scanned,mineru,7,1.0,0.196930,0.300603,0.418632,0.522049,10,0.002381,1.114139,0
5,natively-scanned,marker,7,1.0,0.155997,0.226477,0.591906,0.666555,10,0.007034,34.744778,0
6,simulated-scanned,docling,95,1.0,0.260856,0.305621,0.354479,0.423783,127,0.074761,2.580174,0
7,simulated-scanned,mineru,95,1.0,0.147338,0.218956,0.317217,0.380749,127,0.038877,1.083013,0
8,simulated-scanned,marker,95,1.0,0.112352,0.182288,0.463534,0.529235,127,0.023751,17.148162,0


| Condition         |   n | Tool    |   NED ↓ |   CER ↓ |   TEDS ↑ |   TEDS-S ↑ |   Read. order ↓ |   s/page ↓ |   Success ↑ |
|:------------------|----:|:--------|--------:|--------:|---------:|-----------:|----------------:|-----------:|------------:|
| digital-born      |  87 | docling |  0.1512 |  0.2130 |   0.3819 |     0.4462 |          0.0359 |     0.5895 |      1.0000 |
| digital-born      |  87 | mineru  |  0.1400 |  0.2044 |   0.3138 |     0.3695 |          0.0368 |     0.7292 |      1.0000 |
| digital-born      |  87 | marker  |  0.1159 |  0.1907 |   0.4531 |     0.5281 |          0.0252 |     2.3580 |      1.0000 |
| natively-scanned  |   7 | docling |  0.2583 |  0.2702 |   0.4673 |     0.5398 |          0.1362 |     2.3814 |      1.0000 |
| natively-scanned  |   7 | mineru  |  0.1969 |  0.3006 |   0.4186 |     0.5220 |          0.0024 |     1.1141 |      1.0000 |
| natively-scanned  |   7 | marker  |  0.1560 |  0.2265 |   0.5919 |     0.6666 |          0.0070 |    34.7448 

In [18]:
import subprocess
src = subprocess.run(["find","/kaggle/input","-type","d","-name","results"],capture_output=True,text=True).stdout.strip().split("\n")[0]
!cp -r {src} /kaggle/working/
!find /kaggle/working/results -name "*.json" | wc -l
!ls /kaggle/working/results/docling/meta | wc -l   # expect 95

579
95
